<a href="https://colab.research.google.com/github/taiKip/python-llm-learning/blob/ml/COMP6012_A3_Natural_Intermediate_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMP6012 Assignment 3 — Fine-tuning language models

In this notebook, I'm comparing two models for news classification and two for chatbot responses.

- **AG News:** DistilBERT and RoBERTa
- **PersonaChat:** T5-small and BART-base

I'll test a couple of training settings for each model, compare the results before and after training, and use the graphs and tables in my report.



## 1. Setup

In [ ]:
%pip install -q "transformers<5" peft datasets accelerate rouge-score scikit-learn matplotlib pandas
%pip uninstall -y -q torchao

In [ ]:
import time, gc, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from rouge_score import rouge_scorer
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM,
    Trainer, TrainingArguments, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorWithPadding, DataCollatorForSeq2Seq, set_seed)
from peft import LoraConfig, TaskType, get_peft_model
import transformers, peft, datasets

set_seed(42)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/COMP6012_A3')
except ImportError:
    OUT = Path('./COMP6012_A3')
OUT.mkdir(parents=True, exist_ok=True)

gpu_available = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if gpu_available else 'NONE')
print('Versions:', 'torch', torch.__version__, 'transformers', transformers.__version__,
      'peft', peft.__version__, 'datasets', datasets.__version__)
if not gpu_available:
    print('WARNING: Running on CPU. Training will be extremely slow. Please switch to GPU if possible.')

def save_result(row):
    """Save the latest result for this trial."""
    file = OUT / 'results.csv'
    df = pd.read_csv(file) if file.exists() else pd.DataFrame()
    if not df.empty:
        df = df[~((df.trial == row['trial']) & (df.stage == row['stage']))]
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(file, index=False)
    print(pd.DataFrame([row]).to_string(index=False))

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

(OUT/'environment.json').write_text(json.dumps({
    'gpu': torch.cuda.get_device_name(0) if gpu_available else 'NONE', 'torch': torch.__version__,
    'transformers': transformers.__version__, 'peft': peft.__version__,
    'datasets': datasets.__version__, 'seed': 42}, indent=2))

## 2.  AG News


In [ ]:
ag = load_dataset('fancyzhx/ag_news')
LABELS = ag['train'].features['label'].names
split = ag['train'].train_test_split(test_size=0.05, seed=42, stratify_by_column='label')
news_train, news_val, news_test = split['train'], split['test'], ag['test']
print('Train / validation / test:', len(news_train), len(news_val), len(news_test))
print(pd.Series(ag['train']['label']).map(dict(enumerate(LABELS))).value_counts())
print('Average words:', np.mean([len(s.split()) for s in ag['train']['text']]))
pd.Series(ag['train']['label']).value_counts().sort_index().plot.bar(title='AG News class distribution')
plt.xticks(range(4), LABELS, rotation=20)
plt.tight_layout(); plt.savefig(OUT/'figure1_ag_news.png'); plt.show()

## 3. Models and settings for the news task

 four runs, T1 to T4. `lr` controls the learning rate, `r` is the LoRA rank, and `epochs` tells us how many times the model sees the training data. `batch` is the effective batch size.

In [ ]:
NEWS_MODELS = {'DistilBERT': 'distilbert-base-uncased', 'RoBERTa': 'roberta-base'}
NEWS_TRIALS = [
    dict(trial='T1', model='DistilBERT', lr=2e-4, epochs=1, r=8, batch=32),
    dict(trial='T2', model='DistilBERT', lr=5e-4, epochs=1, r=8, batch=32),
    dict(trial='T3', model='RoBERTa',   lr=2e-4, epochs=1, r=8, batch=32),
    dict(trial='T4', model='RoBERTa',   lr=2e-4, epochs=2, r=16, batch=32),
]
display(pd.DataFrame(NEWS_TRIALS))

### Training and checking the news models


In [ ]:
def news_metrics(prediction):
    logits = prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    guesses = logits.argmax(axis=-1)
    answers = prediction.label_ids
    return {'accuracy': accuracy_score(answers, guesses),
            'macro_f1': f1_score(answers, guesses, average='macro')}

def train_news(cfg):
    set_seed(42)
    free_gpu()
    name = cfg['trial']
    tokenizer = AutoTokenizer.from_pretrained(NEWS_MODELS[cfg['model']])

    def encode(batch):
        return tokenizer(batch['text'], truncation=True, max_length=128)

    def ready(part):
        return part.map(encode, batched=True, remove_columns=['text']).rename_column('label', 'labels')

    train_data, val_data, test_data = ready(news_train), ready(news_val), ready(news_test)
    target = ['q_lin', 'v_lin'] if cfg['model'] == 'DistilBERT' else ['query', 'value']

    # Safely get GPU capabilities only if CUDA is available
    is_cuda = torch.cuda.is_available()
    major = torch.cuda.get_device_capability(0)[0] if is_cuda else 0

    settings = dict(output_dir=str(OUT/name), learning_rate=cfg['lr'],
        num_train_epochs=cfg['epochs'], per_device_train_batch_size=8,
        gradient_accumulation_steps=cfg['batch']//8,
        per_device_eval_batch_size=16, eval_strategy='steps', eval_steps=500,
        logging_steps=100, save_strategy='no', report_to='none',
        fp16=is_cuda and (major < 8),
        bf16=is_cuda and (major >= 8), seed=42)

    def new_model():
        return AutoModelForSequenceClassification.from_pretrained(
            NEWS_MODELS[cfg['model']], num_labels=4,
            id2label=dict(enumerate(LABELS)), label2id={v:k for k,v in enumerate(LABELS)})

    base = new_model()
    baseline = Trainer(model=base, args=TrainingArguments(**settings),
        eval_dataset=val_data, data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=news_metrics)
    before = baseline.evaluate()
    save_result(dict(task='classification', trial=name, model=cfg['model'], stage='before',
        val_loss=before['eval_loss'], val_accuracy=before['eval_accuracy'],
        val_macro_f1=before['eval_macro_f1'], note='Untrained classification head'))
    del baseline, base
    free_gpu()

    set_seed(42)
    model = new_model()
    model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_CLS,
        r=cfg['r'], lora_alpha=2*cfg['r'], lora_dropout=0.05, target_modules=target))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    percent = 100 * trainable / total
    print(f'{name}: {percent:.2f}% trainable parameters')

    trainer = Trainer(model=model, args=TrainingArguments(**settings),
        train_dataset=train_data, eval_dataset=val_data,
        data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=news_metrics)
    start = time.time()
    train_result = trainer.train()
    minutes = (time.time()-start)/60
    after = trainer.evaluate()
    test = trainer.predict(test_data)
    logits = test.predictions[0] if isinstance(test.predictions,tuple) else test.predictions
    predicted = logits.argmax(axis=-1)
    save_result(dict(task='classification', trial=name, model=cfg['model'], stage='after',
        train_loss=train_result.training_loss, val_loss=after['eval_loss'],
        val_accuracy=after['eval_accuracy'], val_macro_f1=after['eval_macro_f1'],
        test_accuracy=accuracy_score(test.label_ids,predicted),
        test_macro_f1=f1_score(test.label_ids,predicted,average='macro'),
        train_minutes=minutes, trainable_pct=percent, lr=cfg['lr'],
        effective_batch=cfg['batch'], epochs=cfg['epochs'],
        lora_r=cfg['r'], lora_alpha=2*cfg['r']))
    np.save(OUT/f'{name}_confusion.npy', confusion_matrix(test.label_ids,predicted,labels=[0,1,2,3]))
    (OUT/f'{name}_log.json').write_text(json.dumps(trainer.state.log_history, indent=2))
    model.save_pretrained(OUT/name/'adapter')
    tokenizer.save_pretrained(OUT/name/'adapter')
    del model,trainer,train_data,val_data,test_data
    free_gpu()
    return name

### Running T1–T4


In [ ]:
# Comment out a trial here if I have already run it.
for trial in NEWS_TRIALS:
    train_news(trial)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(load

## 4. Getting PersonaChat ready

In [ ]:
pc = load_dataset('AlekseyKorshuk/persona-chat')
print(pc)
print('Example keys:', pc['train'][0].keys())

def format_chat(example):
    persona = example['personality']
    turn = example['utterances'][-1]
    history = turn['history'][-4:]
    gold = turn['candidates'][-1]
    prompt = 'persona: ' + ' '.join(persona) + '\nconversation: ' + ' | '.join(history) + '\nreply:'
    return {'input': prompt, 'target': gold}

valid_key = 'validation' if 'validation' in pc else 'valid'
chat_train = pc['train'].map(format_chat, remove_columns=pc['train'].column_names)
chat_val = pc[valid_key].map(format_chat, remove_columns=pc[valid_key].column_names)
chat_train = chat_train.filter(lambda x: bool(x['input'].strip() and x['target'].strip()))
chat_val = chat_val.filter(lambda x: bool(x['input'].strip() and x['target'].strip()))
print('Train / validation:', len(chat_train), len(chat_val))
print(chat_train[0])

## 5. Models and settings for the chatbot

Here I'm comparing T5-small and BART-base. I'll look at validation loss and perplexity, then compare generated replies using ROUGE-L and Distinct-2. To keep comparisons manageable, the reply metrics use the same 100 validation prompts; validation loss uses the full validation set.

In [ ]:
CHAT_MODELS = {'T5': 't5-small', 'BART': 'facebook/bart-base'}
CHAT_TRIALS = [
    dict(trial='G1', model='T5', lr=3e-4, epochs=1, r=8, batch=16),
    dict(trial='G2', model='T5', lr=1e-3, epochs=1, r=8, batch=16),
    dict(trial='G3', model='BART', lr=3e-4, epochs=1, r=8, batch=16),
    dict(trial='G4', model='BART', lr=3e-4, epochs=2, r=16, batch=16),
]
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def reply_metrics(replies, gold):
    rouge_l = np.mean([rouge.score(g,r)['rougeL'].fmeasure for r,g in zip(replies,gold)])
    pairs = [p for text in replies for p in zip(text.split(), text.split()[1:])]
    distinct_2 = len(set(pairs))/len(pairs) if pairs else 0.0
    return {'rougeL': float(rouge_l), 'distinct2': float(distinct_2)}

def make_replies(model, tokenizer, prompts):
    model.eval()
    answers = []
    device = next(model.parameters()).device
    for i in range(0,len(prompts),8):
        batch = tokenizer(prompts[i:i+8], return_tensors='pt', padding=True,
                          truncation=True, max_length=256).to(device)
        with torch.no_grad():
            ids = model.generate(**batch, max_new_tokens=64, do_sample=False)
        answers += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return answers

display(pd.DataFrame(CHAT_TRIALS))

### Training and checking the chatbot models

This is similar to the news training function, except here the models produce text rather than choose a category. I'll save a few example replies so I can compare them in my report. T5 is trained in full precision because it is unstable in fp16; BART uses fp16 on the T4.

In [ ]:
def train_chat(cfg):
    set_seed(42)
    free_gpu()
    name = cfg['trial']
    tokenizer = AutoTokenizer.from_pretrained(CHAT_MODELS[cfg['model']])

    def encode(batch):
        inputs = tokenizer(batch['input'], truncation=True, max_length=256)
        inputs['labels'] = tokenizer(text_target=batch['target'], truncation=True,
                                     max_length=64)['input_ids']
        return inputs

    train_data = chat_train.map(encode, batched=True, remove_columns=['input','target'])
    val_data = chat_val.map(encode, batched=True, remove_columns=['input','target'])
    collator = DataCollatorForSeq2Seq(tokenizer, label_pad_token_id=-100)
    sample = chat_val.select(range(min(100,len(chat_val))))
    prompts, gold = sample['input'], sample['target']

    # Safely get GPU capabilities only if CUDA is available
    is_cuda = torch.cuda.is_available()
    major = torch.cuda.get_device_capability(0)[0] if is_cuda else 0

    use_fp16 = cfg['model'] != 'T5' and is_cuda and major < 8
    use_bf16 = is_cuda and major >= 8
    settings = dict(output_dir=str(OUT/name), learning_rate=cfg['lr'],
        num_train_epochs=cfg['epochs'], per_device_train_batch_size=4,
        gradient_accumulation_steps=cfg['batch']//4,
        per_device_eval_batch_size=8, eval_strategy='steps', eval_steps=200,
        logging_steps=50, save_strategy='no', report_to='none',
        fp16=use_fp16, bf16=use_bf16, seed=42,
        predict_with_generate=False)

    base = AutoModelForSeq2SeqLM.from_pretrained(CHAT_MODELS[cfg['model']])
    base_trainer = Seq2SeqTrainer(model=base, args=Seq2SeqTrainingArguments(**settings),
        eval_dataset=val_data, data_collator=collator, processing_class=tokenizer)
    base_loss = base_trainer.evaluate()['eval_loss']
    before = make_replies(base,tokenizer,prompts)
    save_result(dict(task='generation',trial=name,model=cfg['model'],stage='before',
        val_loss=base_loss, perplexity=float(np.exp(min(base_loss,50))),
        **reply_metrics(before,gold),eval_reply_count=len(prompts)))
    del base,base_trainer
    free_gpu()

    set_seed(42)
    model = AutoModelForSeq2SeqLM.from_pretrained(CHAT_MODELS[cfg['model']])
    model = get_peft_model(model,LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM,
        r=cfg['r'], lora_alpha=2*cfg['r'], lora_dropout=0.05, target_modules=['q', 'v'] if cfg['model'] == 'T5' else ['q_proj', 'v_proj']))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    percent = 100*trainable/total
    trainer = Seq2SeqTrainer(model=model,args=Seq2SeqTrainingArguments(**settings),
        train_dataset=train_data,eval_dataset=val_data,data_collator=collator,
        processing_class=tokenizer)
    start = time.time()
    result = trainer.train()
    minutes = (time.time()-start)/60
    loss = trainer.evaluate()['eval_loss']
    after = make_replies(model,tokenizer,prompts)
    save_result(dict(task='generation',trial=name,model=cfg['model'],stage='after',
        train_loss=result.training_loss,val_loss=loss,perplexity=float(np.exp(min(loss,50))),
        **reply_metrics(after,gold),train_minutes=minutes,trainable_pct=percent,
        lr=cfg['lr'],effective_batch=cfg['batch'],epochs=cfg['epochs'],
        lora_r=cfg['r'],lora_alpha=2*cfg['r'],eval_reply_count=len(prompts)))
    pd.DataFrame({'input':prompts,'gold':gold,'before':before,'after':after}).to_csv(
        OUT/f'{name}_dialogue_examples.csv', index=False)
    (OUT/f'{name}_log.json').write_text(json.dumps(trainer.state.log_history, indent=2))
    model.save_pretrained(OUT/name/'adapter')
    tokenizer.save_pretrained(OUT/name/'adapter')
    del model,trainer,train_data,val_data
    free_gpu()
    return name

### Running G1–G4

These four runs can take time, so I can rerun only the ones I need.

In [ ]:
# I can skip experiments I have already finished.
for trial in CHAT_TRIALS:
    train_chat(trial)

## 6. Comparing the results

Now I'll put the saved scores into tables and plot the training and validation loss. These plots will help me see whether training improved the models or whether they might be overfitting.

In [ ]:
results = pd.read_csv(OUT/'results.csv')
display(results.sort_values(['task','trial','stage']))
for task in ['classification','generation']:
    results[results.task==task].to_csv(OUT/f'table_{task}_results.csv',index=False)

for name in ['T1','T2','T3','T4','G1','G2','G3','G4']:
    path = OUT/f'{name}_log.json'
    if not path.exists():
        continue
    log = json.loads(path.read_text())
    plt.figure()
    for key,label in [('loss','Training'),('eval_loss','Validation')]:
        steps = [(x['step'],x[key]) for x in log if key in x]
        if steps:
            plt.plot(*zip(*steps),label=label,marker='o' if key=='eval_loss' else None)
    plt.title(f'{name}: loss during training')
    plt.xlabel('Training steps');plt.ylabel('Loss');plt.legend()
    plt.tight_layout();plt.savefig(OUT/f'{name}_loss_curve.png');plt.show()

## 7. Confusion matrix for the best news model

I'll pick the classifier using validation macro-F1 and then show its mistakes on the test set. I won't choose the model based on the test scores.

In [ ]:
classification = results[(results.task=='classification') & (results.stage=='after')]
best = classification.sort_values('val_macro_f1',ascending=False).iloc[0]['trial']
cm = np.load(OUT/f'{best}_confusion.npy')
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(cmap='Blues', xticks_rotation=25)
plt.title(f'{best}: test confusion matrix')
plt.tight_layout();plt.savefig(OUT/'figure3_best_confusion.png');plt.show()
print('Best classifier by validation macro-F1:',best)

## 8. Comparing chatbot replies

Automatic scores don't tell the whole story for conversations. I'll also look at example replies and manually rate whether they sound natural, match the persona, and answer the last message. I need to do the ratings myself rather than guess the scores. Note: T5 and BART use different tokenizers, so their validation losses aren't directly comparable; the best chatbot below is picked by loss only as a convenience.

In [ ]:
generation = results[(results.task=='generation') & (results.stage=='after')]
best_gen = generation.sort_values('val_loss').iloc[0]['trial']
examples = pd.read_csv(OUT/f'{best_gen}_dialogue_examples.csv')
display(examples.head(5))
examples.head(10).to_csv(OUT/'table7_example_replies.csv',index=False)

# Make a ratings sheet for 20 examples per model and stage, using each model's best trial,
# with the prompt, gold reply and model reply included so there is something to rate.
rows = []
for model in ['T5','BART']:
    runs = generation[generation.model==model]
    if runs.empty:
        continue
    trial = runs.sort_values('val_loss').iloc[0]['trial']
    replies = pd.read_csv(OUT/f'{trial}_dialogue_examples.csv').head(20)
    for stage in ['before','after']:
        for i,row in enumerate(replies.itertuples(),1):
            rows.append({'model':model,'trial':trial,'stage':stage,'example':i,
                'input':row.input,'gold':row.gold,'reply':getattr(row,stage),
                'human_likeness_1to5':'','persona_consistency_1to5':'','relevance_1to5':''})
pd.DataFrame(rows).to_csv(OUT/'table8_human_ratings_BLANK.csv',index=False)
print('Fill table8_human_ratings_BLANK.csv manually; never report uncollected ratings.')

In [ ]:
# Once I have rated the replies, I can read the averages here.
file = OUT/'table8_human_ratings_COMPLETED.csv'
if file.exists():
    ratings = pd.read_csv(file)
    columns = ['human_likeness_1to5','persona_consistency_1to5','relevance_1to5']
    for c in columns:
        ratings[c] = pd.to_numeric(ratings[c],errors='coerce')
    display(ratings.groupby(['model','stage'])[columns].mean())
else:
    print('No completed ratings yet.')

## 9. Using these results in my report

After running all the experiments, I'll use the saved files to complete the report:

- **Table 1 and Figure 1:** dataset sizes, labels and the AG News chart.
- **Tables 2 and 3:** why I chose each model and which GPU/libraries I used.
- **Table 4:** the different training settings and how long each run took.
- **Tables 5 and 6:** classification and chatbot scores before and after fine-tuning.
- **Figures 2–4:** loss curves and the best classifier's confusion matrix.
- **Table 7:** examples of replies before and after training.
- **Table 8:** my manual ratings of chatbot replies.

**Things to mention in the discussion:** The news models start with an untrained classification head. For PersonaChat, I'm using only the final turn from each example, and my reply metrics use 100 validation prompts. Some runs change more than one setting at once, so I can't say for certain which change caused an improvement. I'll also record the actual Colab GPU and library versions from my run.
